# Image relevance verification

Run every cell in order. These checks use synthetic fixtures and a real disposable PostgreSQL schema. They never print private credentials. Real AI quality is checked only from an explicitly identified live evaluation artifact.

In [1]:
from pathlib import Path
import subprocess, json
ROOT = Path.cwd()
if not (ROOT / "package.json").exists(): ROOT = ROOT.parent
PROOF = ROOT / "docs" / "proof"
PROOF.mkdir(parents=True, exist_ok=True)
def run(args, filename):
    p = subprocess.run(args, cwd=ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=180)
    output = p.stdout + p.stderr
    (PROOF / filename).write_text(output, encoding="utf-8")
    print(output)
    assert p.returncode == 0
    return output
print("Ready: real PostgreSQL, synthetic test providers; no paid calls.")

Ready: real PostgreSQL, synthetic test providers; no paid calls.


In [2]:
api = run(["node", "--test", "--test-reporter=tap", "--test-concurrency=1", *[str(p) for p in sorted((ROOT / "tests").glob("*.test.js"))]], "api-tests.txt")
assert "# fail 0" in api
assert "# pass 15" in api

TAP version 13
# (node:22360) Warning: The 'NO_COLOR' env is ignored due to the 'FORCE_COLOR' env being set.
# (Use `node --trace-warnings ...` to show where the warning was created)
# Subtest: schema rejects malformed vision, nonfinite confidence and invalid vectors
ok 1 - schema rejects malformed vision, nonfinite confidence and invalid vectors
  ---
  duration_ms: 6.4675
  type: 'test'
  ...
# Subtest: guard maps scientific aliases and rejects a wolf even with perfect similarity
ok 2 - guard maps scientific aliases and rejects a wolf even with perfect similarity
  ---
  duration_ms: 12.9506
  type: 'test'
  ...
# Subtest: auth hashes passwords, expires/revokes sessions and rejects unauthenticated access
ok 3 - auth hashes passwords, expires/revokes sessions and rejects unauthenticated access
  ---
  duration_ms: 1451.298
  type: 'test'
  ...
# Subtest: image upload validates real bytes, deduplicates and queues before AI executes
ok 4 - image upload validates real bytes, deduplicates

In [3]:
browser = run(["node", "scripts/browser-proof.js"], "browser-tests.txt")
assert "Browser proof: 3/3 passed" in browser

PASS browser login, authenticated image preview, job progress and cost summary
PASS browser matching, guard explanation and persistent human approval
PASS mobile workspace at 390px without page overflow
Browser proof: 3/3 passed. Synthetic provider only; not model accuracy evidence.
(node:16084) Warning: The 'NO_COLOR' env is ignored due to the 'FORCE_COLOR' env being set.
(Use `node --trace-warnings ...` to show where the warning was created)



## Real-model evidence and submission files
A missing real evaluation is reported as pending; it is never replaced by mock accuracy.

In [4]:
required = ["README.md", "DESIGN.md", "EVIDENCE.md", "BUILDLOG.md", "capstone.yaml", ".env.example", "docs/API.md", "corpus/sources.json", "evals/posts.json"]
assert all((ROOT / p).exists() for p in required)
sources = json.loads((ROOT / "corpus/sources.json").read_text(encoding="utf-8"))
assert len(sources) >= 40
assert all(s.get("license") and s.get("sourcePage") for s in sources)
report = {"api_passed": 15, "browser_passed": 3, "source_images": len(sources), "real_evaluation": "pending", "public_repository": "pending URL"}
real = PROOF / "real-evaluation.json"
if real.exists():
    result = json.loads(real.read_text(encoding="utf-8"))
    assert result["provider"] == "ollama"
    report["real_evaluation"] = {"model": result["visionModel"], "exact_top1": result["test"]["exactTop1"], "coverage": result["test"]["coverage"]}
(PROOF / "verification-summary.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
print("PASS: notebook completed top to bottom; real-model status reported honestly.")

{
  "api_passed": 15,
  "browser_passed": 3,
  "source_images": 40,
  "real_evaluation": "pending",
  "public_repository": "pending URL"
}
PASS: notebook completed top to bottom; real-model status reported honestly.
